##### Imports

In [2]:
# %pip install -r requirements.txt

In [3]:
import pandas as pd
import numpy as np
from urllib.request import urlopen
import certifi
import json
from fredapi import Fred
import os
import ssl

# Custom packages
import derive_features as dd

import warnings
warnings.filterwarnings("ignore")

# Environment variables
import dotenv
dotenv.load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
FMP_API_KEY = os.getenv("FMP_API_KEY")

# Data Collection

### Inputs

In [6]:
def get_jsonparsed_data(url):
    context = ssl.create_default_context(cafile=certifi.where())
    response = urlopen(url, context=context)
    # response = urlopen(url, cafile=certifi.where())
    data = response.read().decode("utf-8")
    return json.loads(data)

idx = pd.read_csv('data/inputs/index_symbols.csv')
comm = pd.read_csv('data/inputs/commodity_symbols.csv')

url = f"https://financialmodelingprep.com/stable/index-list?apikey={FMP_API_KEY}"
fmp_idx = pd.DataFrame(get_jsonparsed_data(url))
fmp_idx = fmp_idx[fmp_idx['symbol'].isin(idx['FMP API Symbol'])].reset_index(drop=True)
fmp_idx['fx_symbol'] = fmp_idx['currency'].apply(lambda x: x+'USD' if x != 'USD' else None)

url = f"https://financialmodelingprep.com/stable/commodities-list?apikey={FMP_API_KEY}"
fmp_comm = pd.DataFrame(get_jsonparsed_data(url))
fmp_comm = fmp_comm[fmp_comm['symbol'].isin(comm['FMP API Symbol'])].reset_index(drop=True)

fmp_idx.to_csv('data/inputs/fmp_index_list.csv', index=False)
fmp_comm.to_csv('data/inputs/fmp_commodity_list.csv', index=False)

date_from = '1990-01-01'
date_to = '2025-10-31'

### Equity Index, Commodity, and FX Daily Timeseries data 

In [11]:
# symbol = fmp_idx.loc[0, 'symbol']
# url = f"https://financialmodelingprep.com/stable/historical-price-eod/full?symbol={symbol}&from={date_from}&to={date_to}&apikey={FMP_API_KEY}"
# df = pd.DataFrame(get_jsonparsed_data(url))\
#     [['symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'vwap']]
# df['date'] = pd.to_datetime(df['date'])
# df = df.set_index('date')
# df.columns = pd.MultiIndex.from_product([[df['symbol'].iloc[0]], df.columns])
# df = df.drop(columns=df.columns[0]).sort_index()

# for i in range(1, len(fmp_idx)):
#     symbol = fmp_idx.loc[i, 'symbol']
#     url = f"https://financialmodelingprep.com/stable/historical-price-eod/full?symbol={symbol}&from={date_from}&to={date_to}&apikey={FMP_API_KEY}"
#     temp = pd.DataFrame(get_jsonparsed_data(url))\
#         [['symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'vwap']]
#     temp['date'] = pd.to_datetime(temp['date'])
#     temp = temp.set_index('date')
#     temp.columns = pd.MultiIndex.from_product([[temp['symbol'].iloc[0]], temp.columns])
#     temp = temp.drop(columns=temp.columns[0]).sort_index()
#     df = pd.concat([df, temp], axis=1)

# msci = pd.read_excel('data/inputs/MSCI_China_Index.xlsx')[:-1]
# msci['Date'] = pd.to_datetime(msci['Date'])
# # msci = msci.reindex(index=df.index)
# msci = msci.set_index('Date')
# msci.columns = pd.MultiIndex.from_product([['MSCI_China'], ['close']])
# msci[('MSCI_China', 'volume')] = np.nan

# df.join(msci, how='left').to_csv('data/processed/index_data.csv')

## -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------###

symbol = fmp_comm.loc[0, 'symbol']
url = f"https://financialmodelingprep.com/stable/historical-price-eod/full?symbol={symbol}&from={date_from}&to={date_to}&apikey={FMP_API_KEY}"
df = pd.DataFrame(get_jsonparsed_data(url))\
    [['symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'vwap']]
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')
df.columns = pd.MultiIndex.from_product([[df['symbol'].iloc[0]], df.columns])
df = df.drop(columns=df.columns[0]).sort_index()

for i in range(1, len(fmp_comm)):
    symbol = fmp_comm.loc[i, 'symbol']
    url = f"https://financialmodelingprep.com/stable/historical-price-eod/full?symbol={symbol}&from={date_from}&to={date_to}&apikey={FMP_API_KEY}"
    temp = pd.DataFrame(get_jsonparsed_data(url))\
        [['symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 'vwap']]
    temp['date'] = pd.to_datetime(temp['date'])
    temp = temp.set_index('date')
    temp.columns = pd.MultiIndex.from_product([[temp['symbol'].iloc[0]], temp.columns])
    temp = temp.drop(columns=temp.columns[0]).sort_index()
    df = pd.concat([df, temp], axis=1)

df.to_csv('data/processed/commodity_data.csv')
# nickel = pd.read_csv('data/inputs/Nickel_futures.csv', parse_dates=['Date'], dayfirst=True, index_col='Date')\
#     .rename_axis('date').sort_index().rename(columns={'Price': 'close', 'Vol.': 'volume', 'Open': 'open', 'High': 'high', 'Low': 'low'})\
#         [['open', 'high', 'low', 'close', 'volume']]
# nickel.index.name = 'date'
# for col in nickel.columns:
#     if nickel[col].dtype == 'object':
#         nickel[col] = pd.to_numeric(nickel[col].astype(str).str.replace(',', ''), errors='coerce')
# nickel.columns = pd.MultiIndex.from_product([['Nickel'], nickel.columns])
# nickel[('Nickel', 'volume')] = np.nan
# df.join(nickel, how='left').to_csv('data/processed/commodity_data.csv')

## -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------###

fx_symbols_retieved = []
symbol = fmp_idx.loc[0, 'fx_symbol']
fx_symbols_retieved.append(symbol)
url = f"https://financialmodelingprep.com/stable/historical-price-eod/full?symbol={symbol}&from={date_from}&to={date_to}&apikey={FMP_API_KEY}"
df = pd.DataFrame(get_jsonparsed_data(url))[['symbol', 'date', 'close']]
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')
df.rename(columns={'close': df['symbol'].iloc[0]}, inplace=True)
df = df.drop(columns=df.columns[0]).sort_index()

for i in range(1, len(fmp_idx)):
    symbol = fmp_idx.loc[i, 'fx_symbol']
    if symbol is None or symbol in fx_symbols_retieved:
        continue
    fx_symbols_retieved.append(symbol)
    url = f"https://financialmodelingprep.com/stable/historical-price-eod/full?symbol={symbol}&from={date_from}&to={date_to}&apikey={FMP_API_KEY}"
    temp = pd.DataFrame(get_jsonparsed_data(url))[['symbol', 'date', 'close']]
    temp['date'] = pd.to_datetime(temp['date'])
    temp = temp.set_index('date')
    temp.rename(columns={'close': temp['symbol'].iloc[0]}, inplace=True)
    temp = temp.drop(columns=temp.columns[0]).sort_index()
    df = pd.concat([df, temp], axis=1)

df.to_csv('data/processed/fx_data.csv')

### Macroeconomic Data

In [17]:
# Core macro and rate series
macro_codes_dict = {
    # RATE BENCHMARKS
    'DFF': 'Federal Funds Effective Rate',
    'SOFR': 'Secured Overnight Financing Rate',
    # TREASURY RATES
    'DGS1MO': '1-Month Treasury Rate',
    'DGS3MO': '3-Month Treasury Rate', 
    'DGS6MO': '6-Month Treasury Rate',
    'DGS1': '1-Year Treasury Rate',
    'DGS2': '2-Year Treasury Rate',
    'DGS3': '3-Year Treasury Rate',
    'DGS5': '5-Year Treasury Rate',
    'DGS7': '7-Year Treasury Rate',
    'DGS10': '10-Year Treasury Rate',
    'DGS20': '20-Year Treasury Rate',
    'DGS30': '30-Year Treasury Rate',
    # OTHER RATES
    'DAAA': 'Moody\'s Seasoned AAA Corporate Bond Yield',
    'DBAA': 'Moody\'s Seasoned BAA Corporate Bond Yield',
    'OBMMCONF30YF': '30-Year Fixed Rate Conforming Mortgage Index',
    'DPRIME': 'Bank Prime Loan Rate',
    'T5YIE': '5-Year Breakeven Inflation Rate',
    'T10YIE': '10-Year Breakeven Inflation Rate',
    'T30YIE': '30-Year Breakeven Inflation Rate',
    # ECONOMIC INDICATORS
    'GDP': 'Gross Domestic Product',
    'GDPC1': 'Real Gross Domestic Product',
    'A939RX0Q048SBEA': 'Real GDP Per Capita',
    'PCE': 'Personal Consumption Expenditures',
    'PCEPI': 'Personal Consumption Expenditures Price Index',
    'PCEC96': 'Real Personal Consumption Expenditures',
    'CPIAUCSL': 'Consumer Price Index',
    'CPILFESL': 'Core CPI (Less Food and Energy)',
    'UNRATE': 'Unemployment Rate',
    'CIVPART': 'Labor Force Participation Rate',
    'INDPRO': 'Industrial Production Index',
    'PAYEMS': 'Total Nonfarm Payrolls',
    'HOUST': 'Housing Starts',
    'PERMIT': 'Building Permits',
    'MTSDS133FMS': 'Monthly US Government Surplus/Deficit',
    'GFDEGDQ188S': 'Federal Government Debt to GDP Ratio',
    'PMSAVE': 'Personal Savings',
    'PSAVERT': 'Personal Saving Rate',
    'GPDI': 'Gross Private Domestic Investment',
    'GPDIC1': 'Real Gross Private Domestic Investment',
    'BOGZ1FU263092001Q': 'Foreign Direct Investment in the United States',
    'QBPBSTAS': 'Balance Sheet - Total Assets',
    'FDHBFRBN': 'Federal Debt Held by Federal Reserve Banks',
    'FYGFDPUN': 'Federal Debt Held by the Public',
    'FDHBFIN': 'Federal Debt Held by Foreign Investors',
    'COMPOUT': 'Commercial Paper Outstanding',
    'ABCOMP': 'Asset-Backed Commercial Paper Outstanding',
    # MONEY SUPPLY
    'M1SL': 'M1 Money Stock',
    'M2SL': 'M2 Money Stock',
    'BASE': 'St. Louis Adjusted Monetary Base',
    # MARKET INDICATORS
    'VIXCLS': 'CBOE Volatility Index (VIX)',
    'UMCSENT': 'University of Michigan Consumer Sentiment',
    'USSLIND': 'Leading Index for the United States',
    'VISASMIHSA': 'Visa U.S. Consumer Spending Momentum Index: Headline',
    'VISASMIDSA': 'Visa U.S. Consumer Spending Momentum Index: Discretionary',
}

macro_units = {
    # INTEREST RATES AND FINANCIAL RATES
    'DFF': 'Percent, Seasonally Adjusted',
    'SOFR': 'Percent, Not Seasonally Adjusted',
    'DGS1MO': 'Percent, Not Seasonally Adjusted',
    'DGS3MO': 'Percent, Not Seasonally Adjusted',
    'DGS6MO': 'Percent, Not Seasonally Adjusted',
    'DGS1': 'Percent, Not Seasonally Adjusted',
    'DGS2': 'Percent, Not Seasonally Adjusted',
    'DGS3': 'Percent, Not Seasonally Adjusted',
    'DGS5': 'Percent, Not Seasonally Adjusted',
    'DGS7': 'Percent, Not Seasonally Adjusted',
    'DGS10': 'Percent, Not Seasonally Adjusted',
    'DGS20': 'Percent, Not Seasonally Adjusted',
    'DGS30': 'Percent, Not Seasonally Adjusted',
    'DAAA': 'Percent, Not Seasonally Adjusted',
    'DBAA': 'Percent, Not Seasonally Adjusted',
    'OBMMCONF30YF': 'Percent, Not Seasonally Adjusted',
    'DPRIME': 'Percent, Not Seasonally Adjusted',
    'T5YIE': 'Percent, Not Seasonally Adjusted',
    'T10YIE': 'Percent, Not Seasonally Adjusted',
    'T30YIE': 'Percent, Not Seasonally Adjusted',
    # ECONOMIC INDICATORS
    'GDP': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'GDPC1': 'Billions of Chained 2017 Dollars, Seasonally Adjusted Annual Rate',
    'A939RX0Q048SBEA': 'Chained 2017 Dollars, Seasonally Adjusted',
    'PCE': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'PCEPI': 'Index 2017=100, Seasonally Adjusted',
    'PCEC96': 'Billions of Chained 2017 Dollars, Seasonally Adjusted',
    'CPIAUCSL': 'Index 1982-1984=100, Seasonally Adjusted',
    'CPILFESL': 'Index 1982-1984=100, Seasonally Adjusted',
    'UNRATE': 'Percent, Seasonally Adjusted',
    'CIVPART': 'Percent, Seasonally Adjusted',
    'INDPRO': 'Index 2017=100, Seasonally Adjusted',
    'PAYEMS': 'Thousands of Persons, Seasonally Adjusted',
    'HOUST': 'Thousands of Units, Seasonally Adjusted Annual Rate',
    'PERMIT': 'Thousands of Units, Seasonally Adjusted Annual Rate',
    'MTSDS133FMS': 'Millions of Dollars, Not Seasonally Adjusted',
    'GFDEGDQ188S': 'Percent of GDP, Not Seasonally Adjusted',
    'PMSAVE': 'Billions of Dollars, Not Seasonally Adjusted',
    'PSAVERT': 'Percent, Seasonally Adjusted',
    'GPDI': 'Billions of Dollars, Seasonally Adjusted Annual Rate',
    'GPDIC1': 'Billions of Chained 2017 Dollars, Seasonally Adjusted Annual Rate',
    'BOGZ1FU263092001Q': 'Millions of Dollars, Not Seasonally Adjusted',
    'QBPBSTAS': 'Millions of Dollars, Not Seasonally Adjusted',
    'FDHBFRBN': 'Billions of Dollars, Not Seasonally Adjusted',
    'FYGFDPUN': 'Millions of Dollars, Not Seasonally Adjusted',
    'FDHBFIN': 'Billions of Dollars, Not Seasonally Adjusted',
    'COMPOUT': 'Billions of Dollars, Not Seasonally Adjusted',
    'ABCOMP': 'Billions of Dollars, Not Seasonally Adjusted',
    # MONEY SUPPLY
    'M1SL': 'Billions of Dollars, Seasonally Adjusted',
    'M2SL': 'Billions of Dollars, Seasonally Adjusted',
    'BASE': 'Millions of Dollars, Not Seasonally Adjusted',
    # MARKET INDICATORS
    'VIXCLS': 'Index, Not Seasonally Adjusted',
    'UMCSENT': 'Index 1966:Q1=100, Not Seasonally Adjusted',
    'USSLIND': 'Percent, Seasonally Adjusted',
    'VISASMIHSA': 'Index, Seasonally Adjusted',
    'VISASMIDSA': 'Index, Seasonally Adjusted',
}


In [18]:
def get_comprehensive_macro_data(fred_api_key, series_dict=macro_codes_dict, start_date='1990-01-01', end_date='2025-10-24'):
    fred = Fred(api_key=fred_api_key)
    all_data = pd.DataFrame(index=pd.DatetimeIndex(pd.date_range(start=start_date, end=end_date, freq='D')))
    successful_series = []
    failed_series = []
    
    for code, description in series_dict.items():
        try:
            if code in ['MORTGAGE30US', 'BASE']:
                series_data = fred.get_series(code, obervation_satrt=start_date, frequency='m', aggregation_method='eop').rename(code)
            else:
                series_data = fred.get_series(code, observation_start=start_date).rename(code)
            if not series_data.empty:
                all_data = all_data.join(series_data, how='left')
                successful_series.append((code, description))
                # print(f"✓ {code}: {len(series_data)} observations")
            else:
                failed_series.append((code, "No data in time range"))
                # print(f"✗ {code}: No data in specified time range")
        except Exception as e:
            failed_series.append((code, str(e)))
            # print(f"✗ {code}: {e}")

    all_data = all_data.sort_index()
    all_data.index.name = 'date'
    # print(f"Final dataset: {all_data.shape[0]} observations, {all_data.shape[1]} variables")
    # print(f"Date range: {all_data.index.min()} to {all_data.index.max()}")
    return all_data, successful_series, failed_series

macro_data, successful, failed = get_comprehensive_macro_data(FRED_API_KEY, macro_codes_dict, start_date=date_from, end_date=date_to)

# Separate columns by their data frequency (daily, monthly, quarterly, etc.)
def infer_frequency(series):
    # Drop NaNs and get sorted index
    idx = series.dropna().index
    if len(idx) < 2:
        return 'unknown'
    # Calculate median difference in days
    freq_days = (idx[1:] - idx[:-1]).days
    median_days = np.median(freq_days)
    if median_days <= 2:
        return 'daily'
    elif 25 <= median_days <= 35:
        return 'monthly'
    elif 80 <= median_days <= 100:
        return 'quarterly'
    elif 350 <= median_days <= 370:
        return 'yearly'
    else:
        return 'other'

frequency_map = {}
for col in macro_data.columns:
    frequency_map[col] = infer_frequency(macro_data[col])

daily_cols = [col for col, freq in frequency_map.items() if freq == 'daily']
monthly_cols = [col for col, freq in frequency_map.items() if freq == 'monthly']
quarterly_cols = [col for col, freq in frequency_map.items() if freq == 'quarterly']
# yearly_cols = [col for col, freq in frequency_map.items() if freq == 'yearly']
# other_cols = [col for col, freq in frequency_map.items() if freq == 'other']
# unknown_cols = [col for col, freq in frequency_map.items() if freq == 'unknown']

macro_data_daily = macro_data[daily_cols].dropna(how='all', axis=0)
macro_data_monthly = macro_data[monthly_cols].dropna(how='all', axis=0)
macro_data_quarterly = macro_data[quarterly_cols].dropna(how='all', axis=0)

macro_data_daily.to_csv('data/processed/macro_data_daily.csv')
macro_data_monthly.to_csv('data/processed/macro_data_monthly.csv')
macro_data_quarterly.to_csv('data/processed/macro_data_quarterly.csv')